# Genre and movie analysis

This notebook uses the leader's exploded_genres and rating_genre_df tables. Multi-genre movies contribute to each constituent genre, and every result reports support.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from load_data import load_movielens
from preprocess import build_shared_tables
from metrics import genre_pair_stats, genre_q1_table, genre_summary
data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
try:
    frames = load_movielens(data_dir)
except FileNotFoundError:
    frames = load_movielens(PROJECT_ROOT / 'data' / 'sample')
tables = build_shared_tables(frames['ratings'], frames['movies'], frames.get('tags'))
ratings = tables['ratings']
movies = tables['movies']
movie_stats = tables['movie_stats']
exploded_genres = tables['exploded_genres']
rating_genre_df = tables['rating_genre_df']

## Genre summary and Challenge Question 1

In [ ]:
genre_stats = genre_summary(exploded_genres, rating_genre_df)
q1 = genre_q1_table(movie_stats, rating_genre_df, exploded_genres, min_movie_ratings=1000)
display(genre_stats)
display(q1)
print('Always report movie_count and rating_count next to average rating.')

## Multi-genre comparison and Challenge Question 5

In [ ]:
single_multi = (
    movie_stats.merge(ratings[['movieId', 'rating']], on='movieId', how='inner')
    .groupby('is_multigenre', as_index=False)
    .agg(movie_count=('movieId', 'nunique'), rating_count=('rating', 'size'), avg_rating=('rating', 'mean'), median_rating=('rating', 'median'))
)
pair_stats = genre_pair_stats(ratings, movies, genre_stats, min_movie_count=50)
display(single_multi)
display(pair_stats.head(20))
if pair_stats.empty:
    print('No pair meets the primary minimum-support rule in this run.')

The pair-versus-single comparison is descriptive because individual genre averages overlap with the pair movies. A low-support pair must not be presented as a robust finding.